In [ ]:
# ============================================================
# ADVANCED TIME SERIES ANALYSIS
# Project: Forecasting Household Deposit Volume in Russia
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# BLOCK 1. DATA LOADING AND BRIEF EDA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded")

# 1.1. Load data
url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')

# Convert dates
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"Period: from {df.index.min()} to {df.index.max()}")

# 1.2. Brief EDA (as a reminder)
print("\n" + "="*60)
print("BRIEF EDA (reminder)")
print("="*60)

print("\n📊 First 5 rows:")
print(df.head())

print("\n📊 Descriptive statistics (DEPOS):")
print(df['DEPOS'].describe())

# Visualization of DEPOS (quick plot)
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['DEPOS'], color='steelblue', linewidth=1.5)
plt.title('Household Deposit Volume in Russia (2014–2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposit volume, billion RUB')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('02_depos_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Block 1 completed")

# ============================================================
# BLOCK 2. ADVANCED TIME SERIES ANALYSIS
# ============================================================

print("\n" + "="*60)
print("BLOCK 2. ADVANCED TIME SERIES ANALYSIS")
print("="*60)

# 2.1. Time series decomposition (additive)
print("\n🔍 Time series decomposition (additive model)...")

decomposition = seasonal_decompose(df['DEPOS'], model='additive', period=12)

# Visualization of components
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

axes[0].plot(df.index, df['DEPOS'], color='steelblue', linewidth=1.5)
axes[0].set_title('Original series (DEPOS)', fontsize=12)
axes[0].set_ylabel('billion RUB')
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition.trend, color='darkgreen', linewidth=1.5)
axes[1].set_title('Trend', fontsize=12)
axes[1].set_ylabel('billion RUB')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomposition.seasonal, color='darkorange', linewidth=1.5)
axes[2].set_title('Seasonal component', fontsize=12)
axes[2].set_ylabel('billion RUB')
axes[2].grid(True, alpha=0.3)

axes[3].plot(decomposition.resid, color='darkred', linewidth=1.5)
axes[3].set_title('Residuals (noise)', fontsize=12)
axes[3].set_ylabel('billion RUB')
axes[3].set_xlabel('Date')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Time Series Decomposition of DEPOS (additive model)', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('02_decomposition_additive.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Decomposition completed")
print("   📌 Seasonal component extracted using moving average (period=12)")

# 2.2. Seasonality analysis: average values and residuals by month
print("\n🔍 Seasonality analysis...")

# Create a copy for analysis
df_seasonal = df.copy()
df_seasonal['Month'] = df_seasonal.index.month
df_seasonal['Trend'] = decomposition.trend
df_seasonal['Seasonal'] = decomposition.seasonal
df_seasonal['Resid'] = decomposition.resid

# Average values by month
monthly_avg = df_seasonal.groupby('Month')['DEPOS'].mean()
monthly_seasonal = df_seasonal.groupby('Month')['Seasonal'].mean()

# Create table (for collapsible block)
seasonal_table = pd.DataFrame({
    'Average value (billion RUB)': monthly_avg.round(1),
    'Seasonal component (billion RUB)': monthly_seasonal.round(1)
})

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
seasonal_table.index = month_names

# Seasonal component statistics
peak_month = monthly_seasonal.idxmax()
peak_value = monthly_seasonal.max()
peak_avg = monthly_avg[peak_month]
low_month = monthly_seasonal.idxmin()
low_value = monthly_seasonal.min()
low_avg = monthly_avg[low_month]
range_seasonal = peak_value - low_value

# Month names
peak_name = month_names[peak_month - 1]
low_name = month_names[low_month - 1]

# Average deposit volume
mean_depos = df['DEPOS'].mean()
seasonal_percent = (range_seasonal / mean_depos) * 100

# Print main results (compactly)
print("\n📊 SEASONALITY (key indicators):")
print(f"   🔺 PEAK: {peak_name} (+{peak_value:.1f} billion RUB)")
print(f"      Average value in this month: {peak_avg:.1f} billion RUB")
print(f"   🔻 MINIMUM: {low_name} ({low_value:.1f} billion RUB)")
print(f"      Average value in this month: {low_avg:.1f} billion RUB")
print(f"   📊 Range of seasonal fluctuations: {range_seasonal:.1f} billion RUB")
print(f"      This constitutes {seasonal_percent:.2f}% of the average deposit volume ({mean_depos:.0f} billion RUB)")

# Detailed table (hidden by default, expands on request)
from IPython.display import HTML, display

table_html = seasonal_table.to_html()

print("\n📋 Detailed seasonality table by month:")
display(HTML(f"""

    Click to expand
    {table_html}

"""))

# Visualization of seasonal component
plt.figure(figsize=(12, 5))
monthly_seasonal.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Seasonal Component of DEPOS by Month (residuals from trend)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Seasonal component, billion RUB')
plt.xticks(rotation=0)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('02_seasonal_component.png', dpi=300, bbox_inches='tight')
plt.show()

# 2.3. Stationarity check (ADF test)
print("\n🔍 Stationarity check (ADF test)...")

def adf_test(series, series_name):
    result = adfuller(series, autolag='AIC')
    print(f"\n  {series_name}:")
    print(f"    ADF statistic: {result[0]:.4f}")
    print(f"    p-value: {result[1]:.4f}")
    print(f"    Conclusion: {'Stationary ✅' if result[1] < 0.05 else 'Non-stationary ❌'}")

adf_test(df['DEPOS'], 'DEPOS')

# 2.4. Chow test for structural break
print("\n🔍 Chow test for structural break...")

df_before = df[df.index < '2022-01-01'].copy()
df_after = df[df.index >= '2022-01-01'].copy()

df_before['t'] = range(len(df_before))
df_after['t'] = range(len(df_after))

def calculate_rss(df, dep_var, ind_var):
    from sklearn.linear_model import LinearRegression
    X = df[[ind_var]].values
    y = df[dep_var].values
    model = LinearRegression().fit(X, y)
    residuals = y - model.predict(X)
    return np.sum(residuals**2)

df_combined = pd.concat([df_before, df_after])
df_combined['t'] = range(len(df_combined))
rss_combined = calculate_rss(df_combined, 'DEPOS', 't')

rss_before = calculate_rss(df_before, 'DEPOS', 't')
rss_after = calculate_rss(df_after, 'DEPOS', 't')
rss_separate = rss_before + rss_after

k = 2
n = len(df_combined)

f_stat = ((rss_combined - rss_separate) / k) / (rss_separate / (n - 2*k))
p_value = 1 - stats.f.cdf(f_stat, k, n - 2*k)

print(f"\n📊 Chow test results:")
print(f"   F-statistic: {f_stat:.4f}")
print(f"   p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\n📌 CONCLUSION: Structural break is statistically significant (p < 0.05)")
    print("   → Recommended to add a dummy variable for the period after 2022")
else:
    print("\n📌 CONCLUSION: Structural break is not confirmed (p >= 0.05)")

# 2.5. Visualization of structural break
df_before_plot = df[df.index < '2022-01-01']
df_after_plot = df[df.index >= '2022-01-01']

plt.figure(figsize=(14, 6))
plt.plot(df_before_plot.index, df_before_plot['DEPOS'],
         color='steelblue', linewidth=1.5, label='Before 2022')
plt.plot(df_after_plot.index, df_after_plot['DEPOS'],
         color='darkorange', linewidth=1.5, label='Since 2022')

z1 = np.polyfit(range(len(df_before_plot)), df_before_plot['DEPOS'], 1)
p1 = np.poly1d(z1)
plt.plot(df_before_plot.index, p1(range(len(df_before_plot))),
         'r--', linewidth=1.5, alpha=0.7, label=f'Trend before 2022 (slope={z1[0]:.1f})')

z2 = np.polyfit(range(len(df_after_plot)), df_after_plot['DEPOS'], 1)
p2 = np.poly1d(z2)
plt.plot(df_after_plot.index, p2(range(len(df_after_plot))),
         'g--', linewidth=1.5, alpha=0.7, label=f'Trend since 2022 (slope={z2[0]:.1f})')

plt.title('Trend Comparison: Before and After 2022', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposit volume, billion RUB')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('02_structural_break_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Trend slope:")
print(f"   Before 2022: {z1[0]:.2f} billion RUB/month")
print(f"   Since 2022: {z2[0]:.2f} billion RUB/month")
print(f"   Change: {((z2[0] - z1[0]) / z1[0] * 100):.1f}%")

print("\n✅ Block 2 completed")